# Kaggle GPU Training - Real-Time Multimodal Incident Intelligence System

This notebook trains the anomaly pipeline and exports model artifacts for reuse in the project backend.

## Dataset Choice

Primary dataset: **UCF Crime Dataset** on Kaggle (`odins0n/ucf-crime-dataset`).

Why this one:
- Closely matches surveillance incident detection tasks.
- Includes public-safety classes such as RoadAccidents, Robbery, Fighting, Assault, etc.
- Manageable size for Kaggle sessions compared with full raw-video mirrors.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print('CUDA available:', gpu_available)
if gpu_available:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training will run on CPU')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/saksham-1304/VIGIL-AI-Visual-Intelligence-Global-Incident-Learning'
REPO_DIR = Path('/kaggle/working/incident-intel-repo')

if not REPO_DIR.exists():
    if REPO_URL == 'YOUR_GITHUB_REPO_URL':
        raise ValueError('Set REPO_URL to your GitHub repository URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'ml/requirements.txt'], check=True)

In [ ]:
from pathlib import Path
import subprocess
import sys

DATASET_DIR = Path('/kaggle/input/ucf-crime-dataset')
OUTPUT_DIR = Path('/kaggle/working/incident-intel-output')

if not DATASET_DIR.exists():
    raise FileNotFoundError('Expected Kaggle input at /kaggle/input/ucf-crime-dataset. Add the dataset in notebook settings.')

cmd = [
    sys.executable, 'scripts/kaggle_train.py',
    '--input-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'auto',
    '--epochs', '40',
    '--latent-dim', '64',
    '--batch-size', '128',
    '--frame-step', '6',
    '--max-images', '300000'
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

output_dir = Path('/kaggle/working/incident-intel-output')
print('Output files:')
for p in sorted(output_dir.glob('*')):
    print('-', p.name, f'({p.stat().st_size/1024/1024:.2f} MB)')

summary_path = output_dir / 'training_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nBest model:', summary.get('best_model'))
    print('Runtime:', summary.get('runtime'))

## After Training

1. Download `/kaggle/working/incident-intel-output/incident_intel_training_outputs.zip`.
2. Keep heavy binaries out of Git commits by default.
3. Publish artifacts using GitHub Releases or DVC remote storage.